# Clasificación entre Perros y Gatos

In [ ]:
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2 as transforms
from torchvision.models import resnet34, vgg16
from torchvision.io import decode_image
from tqdm import tqdm
from torchinfo import summary

from torchmetrics.classification import BinaryAccuracy, BinaryF1Score, BinaryAUROC
from sklearn import metrics

# Carpeta raiz de los datos
datafolder = "datasets/cats-vs-dogs"

Primero, vamos a comprobar la GPU disponible

In [ ]:
# revisar GPU disponible

if torch.cuda.is_available():
    str_device = "cuda"
elif torch.backends.mps.is_available():
    str_device = "mps"
elif torch.xpu.is_available():
    str_device = "xpu"
else:
    str_device = "cpu"

print(f"Acelerador {str_device} disponible")
device = torch.device(str_device)


## Definición de hiperparametros

In [ ]:
# parametros de entrenamiento
lr = 1e-3
batch_size = 32
epochs = 10
image_size = (224, 224)


## Carga de imagenes
Las imagenes se encuentran dentro de la carpeta del conjunto, organizadas por `[especie].#.jpg`, donde `[especie] = "cat", "dog"`

Para cargarlas, crearemos un objeto que nos permita procesar las imagenes

In [ ]:
# definimos las rutas de los datos de entrenamiento y test
train_dir = os.path.join(datafolder, "train")
test_dir = os.path.join(datafolder, "test1")

In [ ]:
# Objeto Dataset personalizado para cargar las imágenes


class CatsDogsDataset(Dataset):
    def __init__(
        self, datadir, split="train", val_split=0.0, seed=1234, transform=None
    ):
        self.datadir = datadir
        self.split = split
        self.val_split = val_split
        self.transform = transform
        rng = np.random.default_rng(seed)
        # si es entrenamiento o validación, se separan los datos

        if split in ["train", "val"]:
            files = glob(os.path.join(datadir, "*.jpg"))
            rng.shuffle(files)
            if val_split > 0.0:
                n_include = int(len(files) * (1 - val_split))
                self.files = (
                    files[:n_include] if split == "train" else files[n_include:]
                )
            else:
                self.files = files if split == "train" else []

            print(f"Split {split} con {len(self.files)} imágenes")
        elif split == "test":
            self.files = glob(os.path.join(datadir, "*.jpg"))
            print(f"Split {split} con {len(self.files)} imágenes")
        else:
            raise ValueError(f"Split desconocido: {split}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image_path = self.files[idx]
        imagen = decode_image(image_path, mode="RGB")
        label = (
            torch.tensor([1.0])
            if "dog" in os.path.basename(image_path)
            else torch.tensor([0.0])
        )

        if self.transform:
            imagen = self.transform(imagen)
        return imagen, label

In [ ]:
# transformaciones para el entrenamiento y test
train_transforms = transforms.Compose(
    [
        transforms.ToImage(),
        transforms.Resize(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.ToDtype(torch.float32, scale=True),
    ]
)

test_transforms = transforms.Compose(
    [
        transforms.ToImage(),
        transforms.Resize(image_size),
        transforms.ToDtype(torch.float32, scale=True),
    ]
)


In [ ]:
# Cargamos los datasets y vamos a visualizar algunos ejemplos:

train_dataset = CatsDogsDataset(
    train_dir, split="train", val_split=0.1, transform=train_transforms
)
val_dataset = CatsDogsDataset(
    train_dir, split="val", val_split=0.1, transform=test_transforms
)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(10):
    image, label = train_dataset[i]
    ax = axes[i // 5, i % 5]
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title("Perro" if label == 1 else "Gato")
    ax.axis("off")

plt.show()

## Dataloaders
Pytorch para facilitar el proceso de mezclar y separar los datos, provee DataLoaders.
Los Dataloaders son objetos que permiten automatizar el muestreo de nuestros datos, generando nuestros batches de entrenamiento, mezclar los datos, paralelizar este proceso para acelerar el proceso de creacion de nuestras muestras, etc.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## Modelo y optimizador

Aqui vamos a seleccionar que modelo deseamos entrenar.
Este modelo debemos modificarlo para considerar el total de etiquetas de clase que tendrá. Para este caso, solo será 1

In [ ]:
# Modelo ResNet34 preentrenado

modelo = resnet34(weights="DEFAULT")
modelo.fc = nn.Linear(modelo.fc.in_features, 1)  # Clasificación binaria
modelo = modelo.to(device)

In [ ]:
# Modelo VGG16 preentrenado
modelo = vgg16(weights="DEFAULT")
modelo.classifier[6] = nn.Linear(
    modelo.classifier[6].in_features, 1
)  # Clasificación binaria
modelo = modelo.to(device)

In [ ]:
# Resumen del modelo

summary(
    modelo,
    input_size=(batch_size, 3, *image_size),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    device=device,
)

In [ ]:
# Optimizador y función de pérdida
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(modelo.parameters(), lr=lr)


## Funciones de apoyo

In [ ]:
# función de entrenamiento
def train_epoch(model, loader, criterion, optimizer, device="cpu"):
    model.train()
    total_loss = 0.0

    acc_metric = BinaryAccuracy().to(device)
    f1_metric = BinaryF1Score().to(device)
    auroc_metric = BinaryAUROC().to(device)

    with tqdm(total=len(loader), desc="Entrenando") as pbar:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()  # Limpiar gradientes
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()  # Backpropagation
            optimizer.step()  # Actualizar pesos

            # Actualizar métricas
            acc_metric.update(outputs, labels)
            f1_metric.update(outputs, labels)
            auroc_metric.update(outputs, labels)
            total_loss += loss.item()

            # computamos para mostrar en la barra de progreso
            acc = acc_metric.compute()
            f1 = f1_metric.compute()
            auroc = auroc_metric.compute()
            pbar.set_postfix_str(
                f"Loss: {total_loss / (pbar.n + 1):.4f}, Acc: {acc:.4f}, F1: {f1:.4f}, AUROC: {auroc:.4f}"
            )
            pbar.update(1)
    return total_loss / len(loader)


In [ ]:
def eval_epoch(model, loader, criterion, device="cpu"):
    model.eval()
    total_loss = 0.0

    acc_metric = BinaryAccuracy().to(device)
    f1_metric = BinaryF1Score().to(device)
    auroc_metric = BinaryAUROC().to(device)

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluando"):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            # Actualizar métricas
            acc_metric.update(outputs, labels)
            f1_metric.update(outputs, labels)
            auroc_metric.update(outputs, labels)
            total_loss += loss.item()

    acc = acc_metric.compute()
    f1 = f1_metric.compute()
    auroc = auroc_metric.compute()
    print(
        f"[Eval] Loss: {total_loss / len(loader):.4f}, Acc: {acc:.4f}, F1: {f1:.4f}, AUROC: {auroc:.4f}"
    )
    return total_loss / len(loader)


## ciclo de entrenamiento

In [ ]:
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    train_loss = train_epoch(modelo, train_loader, criterion, optimizer, device)
    val_loss = eval_epoch(modelo, val_loader, criterion, device)

In [ ]:
torch.save(modelo.state_dict(), "saves/modelo_cats_vs_dogs.pth")

## Evaluacion de métricas

In [ ]:
# Obtenemos las predicciones realizadas por el modelo en el set de test
modelo.eval()

all_labels, all_preds, all_probas = [], [], []

# revisamos
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Obteniendo predicciones"):
        images, labels = images.to(device), labels.to(device)
        outputs = modelo(images)
        probas = torch.sigmoid(outputs).detach().cpu().numpy()
        preds = (probas > 0.5).astype(int)

        # acumulamos resultados
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds)
        all_probas.extend(probas)

# verificamos que tengan el mismo largo
assert len(all_labels) == len(all_preds) == len(all_probas), (
    "Los arrays no tienen el mismo largo"
)

In [ ]:
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probas = np.array(all_probas)
print(all_labels.shape, all_preds.shape, all_probas.shape)

In [ ]:
# metricas generales
print(
    metrics.classification_report(all_labels, all_preds, target_names=["Gato", "Perro"])
)

In [ ]:
# matriz de confusion
fig, ax = plt.subplots(figsize=(6, 5))
metrics.ConfusionMatrixDisplay.from_predictions(
    all_labels,
    all_preds,
    display_labels=["Gato", "Perro"],
    cmap="Blues",
    normalize="true",
    ax=ax,
    values_format=".2%",
)
fig.suptitle("Matriz de Confusión")
ax.set_xlabel("Predicción")
ax.set_ylabel("Etiqueta Verdadera")

plt.show()

In [ ]:
# Curva ROC
fig, ax = plt.subplots(figsize=(6, 5))
metrics.RocCurveDisplay.from_predictions(
    all_labels, all_probas, pos_label=1, ax=ax, plot_chance_level=True
)
fig.suptitle("Curva ROC")
ax.set_xlabel("Tasa de Falsos Positivos")
ax.set_ylabel("Tasa de Verdaderos Positivos")
plt.show()

## Evaluación de imagenes de test

Vamos a seleccionar algunas imagenes del set de test (no tienen etiqueta) Y vamos a ver como se desempeña

In [ ]:
def load_image(path):
    imagen = decode_image(path, mode="RGB")
    imagen = test_transforms(imagen)
    return imagen.unsqueeze(0)  # Añadir dimensión de batch


def eval_image(model, image_path, device="cpu"):
    model.eval()
    with torch.no_grad():
        imagen = load_image(image_path)
        imagen = imagen.to(device)
        output = model(imagen)
        proba = torch.sigmoid(output).item()
        pred = 1 if proba > 0.5 else 0
        out_proba = proba if pred == 1 else 1 - proba
    return pred, out_proba

In [ ]:
impath = "datasets/cats-vs-dogs/test1/1.jpg"
imagen = load_image(impath)

fig, axes = plt.subplots(figsize=(4, 4))
axes.imshow(imagen.squeeze().permute(1, 2, 0))
axes.set_title("Imagen de prueba")
axes.axis("off")

In [ ]:
# vamos a seleccionar 10 imagenes aleatorias del path
rand_images = np.random.choice(
    glob(os.path.join(test_dir, "*.jpg")), size=10, replace=False
)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, impath in enumerate(rand_images):
    pred, proba = eval_image(modelo, impath, device)
    imagen = load_image(impath).squeeze().permute(1, 2, 0)
    ax = axes[i // 5, i % 5]
    ax.imshow(imagen)
    ax.set_title(f"{'Perro' if pred == 1 else 'Gato'} ({proba:.2f})")
    ax.axis("off")
plt.show()